# discriminator-classifier-head — worked example 2: Global-average-pool classifier head to a scalar probability

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `discriminator-classifier-head`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

Some discriminators replace the giant flatten+Linear with a lighter head: global-average-pool over spatial dims to get `(B, C)`, then a small `Linear(C, 1)` to one logit, then `Sigmoid`. This keeps far fewer parameters than flattening `C*H*W`, and the pool makes the head agnostic to input resolution. The final scalar is still a single real/fake probability per image.

## Worked solution

**Goal:** map `features: (B, C, H, W)` to `probs: (B,)` using a pooled head.

1. **Global average pool over H and W.** `reduce(features, 'b c h w -> b c', 'mean')` averages each channel map down to one number, giving `(B, C)`. This is the einops way to write `features.mean(dim=(2, 3))`.
2. **Project to a single logit.** `weight` is `(1, C)` and `bias` is `(1,)`, parameterizing `Linear(C, 1)`. Compute `pooled @ weight.T + bias` → `(B, 1)`.
3. **Sigmoid then squeeze.** `t.sigmoid(logits).squeeze(-1)` gives `(B,)` probabilities in `(0, 1)`.

Why this works: averaging over space summarizes "how much of feature k is present anywhere in this image". Because the pool removes H and W, the head's parameter count depends only on C, not on resolution — so the same trained head works on differently sized inputs. The trade-off vs the flatten head is that you discard *where* features fired, which is usually fine at the final classification stage.

In [ ]:
def disc_head_gap(features: Tensor, weight: Tensor, bias: Tensor) -> Tensor:
    pooled = reduce(features, 'b c h w -> b c', 'mean')   # (B, C)
    logits = pooled @ weight.T + bias                     # (B, 1)
    return t.sigmoid(logits).squeeze(-1)                  # (B,)

t.manual_seed(0)
B, C, H, W = 5, 16, 8, 8
features = t.randn(B, C, H, W)
weight = t.randn(1, C)
bias = t.randn(1)
probs = disc_head_gap(features, weight, bias)
print('shape:', tuple(probs.shape))
print('all in (0,1):', bool(((probs > 0) & (probs < 1)).all()))
print('probs:', probs.round(decimals=3).tolist())